##### Change to root for accessing data directory

In [ ]:
import os
from pathlib import Path

current_dir = Path.cwd()

if current_dir.name == "notebooks" and Path.exists(current_dir.parent / Path("data")):
    os.chdir(current_dir.parent)
    print(f"Current directory: {Path.cwd()}")
elif current_dir.name == "dt133g-thesis-project":
    print(f"Current directory: {Path.cwd()}")
else:
    print("Ensure data exists before running this notebook..")

##### Imports and model/data configurations

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import f1_score, roc_auc_score, precision_recall_fscore_support, confusion_matrix, roc_curve, auc

pretrained_models = {
    "cb": "microsoft/codebert-base",
    "gc": "microsoft/graphcodebert-base",
    "ux": "microsoft/unixcoder-base",
    "ct": "Salesforce/codet5p-770m",
    "ct_enc": "Salesforce/codet5p-770m-enc_only",
    "ds": "deepseek-ai/deepseek-coder-1.3b-base",
}

FULL_MODEL_NAME = pretrained_models["ds"]
DATASET_NAME = "droidcollection"

MODEL_NAME = FULL_MODEL_NAME.split("/")[-1].rsplit("-")[0]

FILE_PATH = Path(f"data/fine_tune/models/{DATASET_NAME}/{FULL_MODEL_NAME}/eval_outputs/ood_metrics_{MODEL_NAME}.csv")

print(f"MODEL_NAME: {MODEL_NAME}, FILE_PATH: {FILE_PATH}")

##### Initialize baseline metrics calculation

In [ ]:
def generate_metrics(csv_df):
    """
    Reads the tracking file from disk and calculates baseline metrics
    (Accuracy, Macro F1, ROC-AUC) grouped by model and dataset tier.
    """
    if not Path(FILE_PATH).exists():
        print(f"Target storage file does not exist: {FILE_PATH}")
        return None
    
    # Clean up duplicate header rows caused by multiple append operations
    if "true_label" in csv_df.columns:
        csv_df = csv_df[csv_df["true_label"] != "true_label"].copy()
    
    # Ensure standard structural identification columns exist
    required_cols = ["model_name", "tier", "snippet_id", "true_label", "pred_label", "conf_target"]
    for col in required_cols:
        if col not in csv_df.columns:
            raise ValueError(f"Missing required metric tracking column in CSV: {col}")

    # Strict Type Conversion & Drop Missing Rows
    csv_df = csv_df.dropna(subset=["true_label", "pred_label", "conf_target"])
    
    try:
        csv_df["true_label"] = csv_df["true_label"].astype(float).astype(int)
        csv_df["pred_label"] = csv_df["pred_label"].astype(float).astype(int)
        csv_df["conf_target"] = csv_df["conf_target"].astype(float)
    except ValueError as e:
        print("Data Type Conversion Error! The file likely contains corrupted rows or raw strings in numeric columns.")
        print(f"Details: {e}")
        return None

    # Filter out unlabelled text metrics (true_label == -1)
    eval_df = csv_df[csv_df["true_label"] != -1].copy()
    
    if len(eval_df) == 0:
        print("No valid labeled data rows found in this file to parse.")
        return None

    summary_records = []

    # Group by model and dataset tier
    grouped = eval_df.groupby(["model_name", "tier"])
    
    for (model_name, tier_name), group in grouped:
        print(f"\n" + "="*60)
        print(f"REPORT FOR: {str(model_name).upper()} | TIER: {str(tier_name).upper()}")
        print(f"Evaluated Samples: {len(group)}")
        print("="*60)
        
        y_true = group["true_label"].to_numpy(dtype=np.int32)
        y_pred = group["pred_label"].to_numpy(dtype=np.int32)
        conf_target = group["conf_target"].to_numpy(dtype=np.float64)
        
        # Reconstruct class 1 (Machine Generated) probabilities
        y_prob_machine = np.where(y_true == 1, conf_target, 1.0 - conf_target)

        accuracy = np.mean(y_true == y_pred)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        
        try:
            if len(np.unique(y_true)) > 1:
                roc_auc = roc_auc_score(y_true, y_prob_machine)
                auc_str = f"{roc_auc:.4f}"
            else:
                roc_auc = None
                auc_str = "N/A (Single class slice)"
        except ValueError:
            roc_auc = None
            auc_str = "N/A (Evaluation error)"
            
        precision, recall, f1_by_class, _ = [
            np.asarray(x) for x in precision_recall_fscore_support(
                y_true, y_pred, labels=[0, 1], zero_division=0
            )
        ]
        
        # Print scannable readout
        print(f"Overall Accuracy:  {accuracy:.4f}")
        print(f"Macro F1-Score:    {macro_f1:.4f}")
        print(f"ROC-AUC Score:     {auc_str}")
        print("-" * 40)
        print(f"Class 0 (HUMAN):   Precision: {precision[0]:.4f} | Recall: {recall[0]:.4f} | F1: {f1_by_class[0]:.4f}")
        print(f"Class 1 (MACHINE): Precision: {precision[1]:.4f} | Recall: {recall[1]:.4f} | F1: {f1_by_class[1]:.4f}")
        
        # Append to structural tracker summary
        summary_records.append({
            "model_name": model_name,
            "tier_name": tier_name,
            "sample_count": len(group),
            "accuracy": accuracy,
            "macro_f1": macro_f1,
            "roc_auc": roc_auc,
            "human_f1": f1_by_class[0],
            "machine_f1": f1_by_class[1]
        })
        
    print("\n" + "="*60)
    print("-- All repository evaluations summarized!")
    return pd.DataFrame(summary_records)

##### Read CSV and generate metrics

In [ ]:
df = pd.read_csv(FILE_PATH)
results_df = generate_metrics(df)

##### Plot confusion matrices

In [ ]:
def plot_confusion_matrices(model_name):
    """
    Plots confusion matrices for the thesis evaluation in a structured grid:
    - Row 1: Baseline 'test' set (standalone hero plot)
    - Row 2: tier_1 and tier_2 (side-by-side)
    - Row 3: tier_3 and tier_4 (side-by-side)
    """
    sns.set_theme(style="white")
    
    # Define our structural layout groups
    standalone_tier = "test"
    paired_rows = [
        ["tier_1", "tier_2"],
        ["tier_3", "tier_4"]
    ]
    
    # Helper to pull arrays and calculate the matrix safely
    def get_matrix_data(tier_id):
        # Handle uppercase/lowercase variations in column names dynamically
        t_col = "tier" if "tier" in df.columns else "Tier"
        m_col = "model_name" if "model_name" in df.columns else "Model"
        
        # If the df columns don't have model filters, pull by tier directly
        if m_col in df.columns:
            slice_df = df[(df[m_col] == model_name) & (df[t_col] == tier_id)]
        else:
            slice_df = df[df[t_col] == tier_id] if t_col in df.columns else df
            
        slice_df = slice_df[slice_df["true_label"] != -1]
        if len(slice_df) == 0:
            return None
            
        y_true = slice_df["true_label"].to_numpy(dtype=int)
        y_pred = slice_df["pred_label"].to_numpy(dtype=int)
        return confusion_matrix(y_true, y_pred, labels=[0, 1])

    # Plot baseline matrix (test)
    cm_test = get_matrix_data(standalone_tier)
    if cm_test is not None:
        fig, ax = plt.subplots(figsize=(5.5, 4.5))
        sns.heatmap(cm_test, annot=True, fmt="d", cmap="Blues", square=True,
                    xticklabels=["Human", "Machine"], yticklabels=["Human", "Machine"], ax=ax)
        ax.set_title(f"{model_name.upper()} - Baseline ({standalone_tier.upper()})", fontsize=12, fontweight='bold', pad=10)
        ax.set_ylabel("True Label", fontsize=10)
        ax.set_xlabel("Predicted Label", fontsize=10)
        plt.tight_layout()
        plt.show()
    else:
        print(f"Skipping plot: No data found for baseline '{standalone_tier}'")

    # Plot mutation matrices (2 and 2)
    for row_idx, pairs in enumerate(paired_rows):
        fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
        
        for col_idx, tier_id in enumerate(pairs):
            ax = axes[col_idx]
            cm = get_matrix_data(tier_id)
            
            if cm is not None:
                sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", square=True,
                            xticklabels=["Human", "Machine"], yticklabels=["Human", "Machine"], ax=ax)
                ax.set_title(f"{model_name.upper()} -  {tier_id.upper()}", fontsize=11, fontweight='bold', pad=10)
                ax.set_ylabel("True Label", fontsize=10)
                ax.set_xlabel("Predicted Label", fontsize=10)
            else:
                ax.text(0.5, 0.5, f"No Data for {tier_id}", ha='center', va='center', fontsize=12)
                ax.axis('off')
                
        plt.tight_layout()
        plt.show()
    

In [ ]:
plot_confusion_matrices(MODEL_NAME)

##### Plot ROC curves

In [ ]:
def plot_roc_curves(model_name):
    """
    Plots the ROC curves for a single model across all its evaluated data tiers.
    """
    model_df = df[(df["model_name"] == model_name) & (df["true_label"] != -1)]
    tiers = model_df["tier"].unique()
    
    plt.figure(figsize=(8, 6))
    
    for tier in tiers:
        tier_df = model_df[model_df["tier"] == tier]
        y_true = tier_df["true_label"].to_numpy(dtype=int)
        conf_target = tier_df["conf_target"].to_numpy(dtype=float)
        
        # Reconstruct probability of class 1 (Machine)
        y_prob_machine = np.where(y_true == 1, conf_target, 1.0 - conf_target)
        
        if len(np.unique(y_true)) > 1:
            fpr, tpr, _ = roc_curve(y_true, y_prob_machine)
            roc_auc = auc(fpr, tpr)
            plt.plot(fpr, tpr, label=f"{tier.upper()} (AUC = {roc_auc:.3f})", lw=2)
            
    # Add baseline diagonal guess reference line
    plt.plot([0, 1], [0, 1], color="darkgrey", linestyle="--", label="Random Guess (AUC = 0.500)")
    
    plt.xlim([-0.02, 1.02])
    plt.ylim([-0.02, 1.02])
    plt.title(f"ROC Curves for {model_name.upper()}", fontsize=14, pad=15)
    plt.xlabel("False Positive Rate (FPR)", fontsize=12)
    plt.ylabel("True Positive Rate (TPR)", fontsize=12)
    plt.legend(loc="lower right", fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_roc_curves(MODEL_NAME)

##### Tier order and title map

In [ ]:
sns.set_style("whitegrid")

tiers_order = ["test", "tier_1", "tier_2", "tier_3", "tier_4"]

df["tier"] = pd.Categorical(
    df["tier"],
    categories=tiers_order,
    ordered=True
)

TITLE_MAP = {
    "conf_target": "Predicted Probability",
    "entropy": "Predictive Entropy",
    "epistemic_variance": "Epistemic Variance)"
}

##### Quantile Robustness Gradient

In [ ]:
metrics = [
    "conf_target",
    "entropy",
    "epistemic_variance"
]

quantiles = [0.1, 0.25, 0.5, 0.75, 0.9]

for metric in metrics:

    # Create a new figure for each metric
    plt.figure(figsize=(9, 6))

    summary = (
        df.groupby("tier")[metric]
        .quantile(quantiles)
        .unstack()
        .reindex(tiers_order)
    )

    for q in quantiles:
        plt.plot(
            summary.index,
            summary[q],
            marker="o",
            linewidth=2,
            label=f"q={q}"
        )

    plt.title(f"[{MODEL_NAME.upper()}] Quantile Drift for {TITLE_MAP.get(metric)}", fontsize=14)
    plt.xlabel("Tier")
    plt.ylabel(metric)

    if metric == "epistemic_variance":
        plt.yscale("log")

    plt.legend(title="Quantile", loc="lower right", title_fontsize=12, fontsize=9)
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


##### ECDF Curves

In [ ]:
from statsmodels.distributions.empirical_distribution import ECDF

def plot_ecdfs(df, col, tiers, logx=False, quantile_markers=True, qlist=(0.1,0.5,0.9)):
    # Determine which tiers are actually present and keep order
    present_tiers = [t for t in tiers if t in df['tier'].unique()]
    if not present_tiers:
        raise ValueError("None of the provided tiers are present in the dataframe.")

    colors = sns.color_palette("deep", n_colors=len(present_tiers))

    fig, ax = plt.subplots(figsize=(9,6))
    handles = []
    labels = []

    for i, t in enumerate(present_tiers):
        cum_tiers = present_tiers[:i+1]
        vals = df[df['tier'].isin(cum_tiers)][col].dropna().values
        if vals.size == 0:
            continue

        vals_plot = np.log10(vals + 1e-12) if logx else vals
        ecdf = ECDF(vals_plot)

        # Plot and capture the Line2D handle
        (line,) = ax.plot(ecdf.x, ecdf.y,
                          color=colors[i],
                          linewidth=2.2,
                          marker=None,
                          label=t,
                          zorder=10+i)   # later tiers on top
        handles.append(line)
        labels.append(t)

    # Use the actual handles and nicer labels in legend
    nice_labels = [ {"test":"Base (T0)","tier_1":"Surface (T1)","tier_2":"Natural (T2)",
                     "tier_3":"Adversarial (T3)","tier_4":"Destructive (T4)"}.get(l,l) for l in labels ]
    ax.legend(handles, nice_labels, title="Tier", loc="lower right", title_fontsize=12, fontsize=9)

    # Axis labels
    xlabel_map = {
        "conf_target": "p(true)",
        "entropy": "H(p̄)",
        "epistemic_variance": "log10"
    }
    
    ax.set_xlabel(xlabel_map.get(col, col))
    ax.set_ylabel("Fraction of samples ≤ x")
    ax.set_title(f"[{MODEL_NAME.upper()}] ECDF of {TITLE_MAP.get(col, col)}\n(cumulative tiers)")
    ax.grid(alpha=0.25)
    plt.tight_layout()
    plt.show()


In [ ]:
plot_ecdfs(df, "conf_target", tier_order, logx=False)
plot_ecdfs(df, "entropy", tier_order, logx=False)
plot_ecdfs(df, "epistemic_variance", tier_order, logx=True)